# CI/CD & Monitoring

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 6/6

Shipping a model once is a milestone; shipping safely every week — with machines running the gates and dashboards watching for decay — is a system. This finale wires everything together: GitHub Actions pipelines, release gates that block, rollout patterns that limit blast radius, PSI-based drift detection, and alerts a human actually wants to receive.

## 🎯 Learning Objectives

- Distinguish continuous integration, delivery, deployment and *training* (CT) in an ML setting
- Read a GitHub Actions workflow: triggers, jobs, steps, and where ML gates slot in
- Implement release gates that short-circuit on failing tests, data validation or metrics
- Compare blue-green, canary and shadow deployments; simulate a canary promote-or-rollback call
- Compute Population Stability Index (PSI) and act on its standard thresholds
- Design alert rules that page a human only when a human should be paged

## 1. CI, CD and CT: Three Automations, One Pipeline

| Automation | Runs when | Verifies / delivers |
|---|---|---|
| **Continuous Integration** | every push / PR | code compiles, unit + data + contract tests pass |
| **Continuous Delivery/Deployment** | green CI on main | packaged artefact reaches staging (delivery) or users (deployment) |
| **Continuous Training (CT)** | schedule OR drift trigger | retrains on fresh data, re-runs every gate, registers a candidate |

Classic DevOps stops after CD. ML adds CT because the *world* keeps changing under the
model (Lesson 01): sometimes the right deploy trigger is not a pull request — it is
Thursday, or a PSI alarm.

## 2. Anatomy of a GitHub Actions Workflow

Workflows live in `.github/workflows/*.yml`: a `name`, triggers (`on:`), one or more `jobs`,
each a sequence of `steps` on some runner. For an ML repo, the job is just the Lesson 05
suite plus a metric gate — automation changes *where* tests run, never *whether* they run:

```yaml
name: model-ci                       # shows up in the PR checks list
on:                                  # TRIGGERS: when does this run?
  pull_request: { branches: [main] }
  push: { branches: [main] }

jobs:
  test-and-gate:
    runs-on: ubuntu-latest           # a fresh VM per run - no stale state
    steps:
      - uses: actions/checkout@v4            # 1. clone the repo
      - uses: actions/setup-python@v5        # 2. interpreter
        with: { python-version: "3.12" }
      - run: pip install -r requirements.lock   # 3. PINNED deps (Lesson 02!)
      - run: pytest tests/ -q                # 4. unit + contract tests
      - run: python validate_data.py data/   # 5. data gate (Lesson 05)
      - run: python evaluate.py --min-accuracy 0.90   # 6. metric gate
```

Steps run top-to-bottom; **the first failing step stops the job** and reddens the PR —
that short-circuit *is* the enforcement.

In [ ]:
# The same workflow as a pure-Python gate runner (offline simulation)
def run_workflow(steps: list[tuple[str, bool]]) -> str:
    """Execute steps in order; the FIRST failure stops the job."""
    log = []
    for name, passed in steps:
        mark = "PASS" if passed else "FAIL"
        log.append(f"{mark}  {name}")
        if not passed:
            log.append("JOB STOPPED - red X on the PR")
            return "\n".join(log)
    log.append("ALL GREEN - artefact may be delivered")
    return "\n".join(log)


happy_pipeline = [
    ("checkout + setup-python", True),
    ("pip install -r requirements.lock", True),
    ("pytest tests/ -q", True),
    ("validate_data.py data/", True),
    ("evaluate.py --min-accuracy 0.90", True),
]
broken_data = [("checkout + setup-python", True),
               ("pip install -r requirements.lock", True),
               ("pytest tests/ -q", True),
               ("validate_data.py data/", False)]         # null spike found

print("--- healthy PR ---")
print(run_workflow(happy_pipeline))
print("\n--- PR with poisoned batch ---")
print(run_workflow(broken_data))

## 3. Release Gates in Code

A gate is a predicate with consequences. Collect every signal, require ALL of them, and
return the reasons — because a blocked release nobody understands breeds shadow deploys.

**Syntax:** the all-or-nothing shape:

```python
decision = release_gate(tests=data_tests_ok, data_valid=batch_clean,
                        accuracy=eval_acc, min_accuracy=0.90)
# -> (allowed: bool, reasons: list[str]); CI exits non-zero unless allowed
```

In [ ]:
# One gate, every signal, explicit reasons
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def release_gate(tests_pass: bool, data_valid: bool,
                 accuracy: float, min_accuracy: float = 0.90) -> tuple[bool, list[str]]:
    reasons = []
    if not tests_pass:
        reasons.append("unit/contract tests failed")
    if not data_valid:
        reasons.append("data validation failed")
    if accuracy < min_accuracy:
        reasons.append(f"accuracy {accuracy:.3f} < gate {min_accuracy:.2f}")
    return (not reasons), reasons


candidates = {
    "v5 (current prod)": dict(tests_pass=True, data_valid=True, accuracy=0.912),
    "v6-fast (skipped tests)": dict(tests_pass=False, data_valid=True, accuracy=0.934),
    "v6-clean": dict(tests_pass=True, data_valid=True, accuracy=0.934),
}

for name, signals in candidates.items():
    ok, why = release_gate(**signals)
    verdict = "DEPLOY" if ok else "BLOCKED"
    print(f"{name:<26} -> {verdict:<8} {'| ' + '; '.join(why) if why else ''}")

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          random_state=42, stratify=y)
acc = LogisticRegression(max_iter=2000).fit(X_tr, y_tr).score(X_te, y_te)
ok, _ = release_gate(True, True, acc)
print(f"\nfreshly measured baseline: accuracy {acc:.3f} -> "
      + ("DEPLOY" if ok else "BLOCKED"))

## 4. Rollout Patterns: Limiting Blast Radius

Even a green pipeline can meet a hostile world. Rollout patterns decide how much traffic
risks finding out first:

| Pattern | Mechanism | Best for |
|---|---|---|
| **Blue-green** | two full stacks; switch the router 0->100 instantly | instant rollback, double cost |
| **Canary** | send a small slice (1-10%) to the new model; watch, then widen | statistical safety per taka spent |
| **Shadow** | new model receives copies of traffic, responses discarded | measuring a risky rewrite with zero user risk |

Canary discipline: pre-agree the promotion rule ("canary error rate <= 1.1x stable after N
requests") *before* looking at the data — otherwise every chart argues for whatever you
already wanted.

In [ ]:
# A canary afternoon: 1,000 requests, 5% to the challenger
import numpy as np

rng = np.random.default_rng(7)
N_REQUESTS, CANARY_SHARE = 1000, 0.05

stable_errors = rng.random(N_REQUESTS) < 0.040            # known 4.0% error rate
is_canary = rng.random(N_REQUESTS) < CANARY_SHARE
canary_errors = (rng.random(N_REQUESTS) < 0.046) & is_canary   # challenger: 4.6%

n_canary = int(is_canary.sum())
observed_canary = canary_errors.sum() / max(n_canary, 1)
observed_stable = stable_errors[~is_canary].sum() / int((~is_canary).sum())

print(f"requests routed to canary : {n_canary}")
print(f"observed error rates      : stable {observed_stable:.2%} vs canary {observed_canary:.2%}")

ratio = observed_canary / observed_stable
decision = "PROMOTE -> widen to 50%" if ratio <= 1.1 else "ROLLBACK -> keep stable at 100%"
print(f"canary/stable ratio: {ratio:.2f} (rule: promote at <= 1.10)")
print("DECISION:", decision)

## 5. Watching for Decay: Population Stability Index

Lesson 01 taught you drift exists; PSI *measures* it on the inputs you already log.
Bin the TRAINING distribution, drop today's live values into those same bins, and sum the
signed differences of population shares:

```text
PSI = sum_i (actual_share_i - expected_share_i) * ln(actual_share_i / expected_share_i)
```

| PSI | Reading | Action |
|---|---|---|
| < 0.10 | stable | keep serving |
| 0.10 - 0.25 | moderate shift | investigate, tighten monitoring |
| > 0.25 | major shift | page the owner; consider retrain/repair |

Two habits make PSI trustworthy: bin edges come from TRAINING data (never re-fit on today's
values, or the shift hides itself), and every bin keeps a small epsilon so empty bins cannot
produce `ln(0)`.

In [ ]:
# PSI from scratch: training month vs live month
import numpy as np

EPS = 1e-6


def psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    """Population Stability Index, quantile-binned on the EXPECTED side."""
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf                  # catch outliers
    exp_share = np.histogram(expected, edges)[0] / len(expected)
    act_share = np.histogram(actual, edges)[0] / len(actual)
    exp_share = np.clip(exp_share, EPS, None)
    act_share = np.clip(act_share, EPS, None)
    return float(np.sum((act_share - exp_share) * np.log(act_share / exp_share)))


rng = np.random.default_rng(42)
training_month = rng.normal(50.0, 8.0, 5000)     # e.g. avg support calls, training time

same_world = rng.normal(50.0, 8.0, 1000)         # a calm Tuesday
mild_shift = rng.normal(52.0, 9.0, 1000)         # competitor launched; usage crept up
major_shift = rng.normal(62.0, 14.0, 1000)       # IVR flow changed the field entirely

for label, live in [("calm Tuesday", same_world),
                    ("mild shift", mild_shift),
                    ("major shift", major_shift)]:
    value = psi(training_month, live)
    verdict = ("stable" if value < 0.10 else
               "MODERATE - investigate" if value < 0.25 else "MAJOR - page the owner")
    print(f"PSI({label:<13}) = {value:6.3f}  -> {verdict}")

## 6. Alerting Without Fatigue

Monitoring fails socially before it fails technically: if the channel cries wolf daily, the
real wolf arrives unheard. Route signals by *who must care, how fast*:

| Signal | Threshold example | Route |
|---|---|---|
| Service down / error rate high | 5xx rate > 2% for 5 min | PAGE now |
| Input drift | PSI > 0.25 on a top-3 feature | PAGE the model owner |
| Slow creep | PSI 0.10-0.25 | daily digest ticket |
| Data freshness | no batch by 07:00 | PAGE the data engineer |
| Cost | tokens/day > 2x baseline | weekly review |

Design rule: **every page must be actionable**. If the on-call cannot do anything but read
it, it belongs in the digest, not the pager.

In [ ]:
# Fourteen days of metrics -> pages, tickets, silence
import numpy as np

rng = np.random.default_rng(11)
days = [f"D{d:02d}" for d in range(1, 15)]
psi_top_feature = np.round(np.concatenate([rng.uniform(0.02, 0.08, 9),
                                           [0.28, 0.31],           # drift arrives
                                           rng.uniform(0.05, 0.09, 3)]), 3)
p95_latency_ms = np.round(rng.uniform(95, 130, 14)).astype(int)
freshness_ok = [True] * 6 + [False] + [True] * 7               # one missed ingest


def triage(day, psi_value, latency, fresh):
    if psi_value > 0.25:
        return "PAGE model owner: PSI {:.2f}".format(psi_value)
    if not fresh:
        return "PAGE data engineer: batch missed"
    if latency > 250:
        return "PAGE on-call: p95 {} ms".format(latency)
    if psi_value > 0.10:
        return "ticket: PSI creeping ({:.2f})".format(psi_value)
    return "quiet"


for d, ps, lat, fresh in zip(days, psi_top_feature, p95_latency_ms, freshness_ok):
    print(f"{d}  psi={ps:5.3f}  p95={lat:3d}ms  fresh={str(fresh):<5} -> {triage(d, ps, lat, fresh)}

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Green tests = deploy, no further gates | Broken data and mediocre metrics sail through | Gates must include data validation AND metric thresholds |
| Alerting on everything | Noise trains the team to ignore the channel | Page only on actionable signals; digest the rest |
| Re-fitting PSI bins on today's data | The shift hides itself; PSI stays suspiciously low | Bin edges frozen from TRAINING data |
| PSI on sparse categorical columns | Tiny buckets explode `ln(actual/expected)` | Group rare categories into "Other" first, or use a different statistic |
| Canary judged on 30 requests | One lucky streak promotes (or kills) a fine model | Pre-agree sample size and ratio rule before launch |
| No documented rollback | Incidents begin with improvisation | Every deploy ships with its one-command undo |
| Monitoring infrastructure only | CPU is calm while recall halves | Watch inputs and outputs, not just boxes (Lesson 01's postmortem) |

## 💡 Best Practices & Pro Tips

- **Pre-commit to the decision rule.** Canary thresholds, gate cutoffs, alert levels —
  agreed in daylight, applied at night.
- **Automate the rollback you practiced.** A revert that takes one command gets used; one
  that takes a meeting gets survived-with.
- **Dashboards have audiences**: an executive cares about the KPI, an engineer about p95
  and error budgets. One screen trying to please both pleases neither.
- **Rehearse decay quarterly**: inject a synthetic shift into staging and confirm the alarm
  chain actually fires — untested monitors are decoration.
- **AI-engineering relevance:** for LLM products the same loop runs with new gauges —
  PSI on embedded prompt features, token-cost burn rate as a first-class alert, refusal-rate
  and eval-win-rate as the "recall" of the system. The vendor silently updating the model
  (Lesson 01) is detected by exactly this machinery.
- **Close the loop in the tracker**: when drift triggers a retrain, log it as an experiment
  run tagged with the PSI report (Lessons 03 + 05) — the audit trail is the system.

## 📌 Summary

| Concept / Tool | What it does | Example |
|---|---|---|
| CI workflow (GitHub Actions) | Tests on every PR, first failure stops the job | `pytest` -> `validate_data` -> `evaluate --min-accuracy` |
| Release gate | All-or-nothing deploy predicate with reasons | tests AND data AND accuracy >= 0.90 |
| Canary rollout | Small live slice, pre-agreed promote/rollback rule | 5% of traffic, ratio <= 1.10 |
| Shadow / blue-green | Zero-risk rehearsal / instant switchover | mirror traffic; flip the router |
| PSI | Measures input drift vs training distribution | < 0.10 stable, > 0.25 page |
| Alert routing | Pages for actionable-now, digests for trends | PSI > 0.25 pages; 0.10-0.25 tickets |

Key takeaways:

- CI proves the code; data and metric gates prove the model; CT answers the moving world.
- Blast radius is a design choice: blue-green, canary and shadow trade cost against safety.
- PSI with frozen training bins turns "feels off" into a number with thresholds.
- Monitoring is a social system: pages must be actionable or they teach humans to ignore you.

## 🔗 Next Lesson

Module 18 complete! Take one model you actually care about and walk it the full distance:
pin its environment (Lesson 02), track its experiments (Lesson 03), serve it behind a tested
API (Lesson 04), gate it with data and behaviour tests (Lesson 05), then automate the path
and watch it drift (Lesson 06). The tools will change; the loop will not.